In [1]:
import subprocess

import pandas as pd

import config, utils

# Globals

In [2]:
chm13v2_file_id = "t2t_chm13v2"

# Download and liftover third party data files

Coerce all files to known formats and lift to final reference coordinates.

* CTCF peak data (GRCh38): https://www.encodeproject.org/files/ENCFF797SDL/@@download/ENCFF797SDL.bed.gz
* GM12878 expression data (GRCh38): https://www.encodeproject.org/files/ENCFF978HIY/@@download/ENCFF978HIY.tsv
* GM12878 ATAC-seq data (GRCh38): https://www.encodeproject.org/files/ENCFF748UZH/@@download/ENCFF748UZH.bed.gz
* TSS location data: TODO: ADD THIS
* LAD data: TODO: ADD THIS 
* Chain files:
  * https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/chain/v1_nflo/grch38-chm13v2.chain
  * https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/chain/v1_nflo/hg19-chm13v2.chain
* Genome assembly: https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/analysis_set/chm13v2.0.fa.gz

In [ ]:
t2t_reference = utils.save_file_from_url("https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/analysis_set/chm13v2.0.fa.gz")
subprocess.run(["samtools", "faidx", t2t_reference])

CompletedProcess(args=['samtools', 'faidx', '/Users/jeremy/devspace/multimodal-dimelo/data/raw/chm13v2.0.fa'], returncode=0)

In [ ]:
grch38_chm13v2_chain_file = utils.save_file_from_url("https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/chain/v1_nflo/grch38-chm13v2.chain")
hg19_chm13v2_chain_file = utils.save_file_from_url("https://s3-us-west-2.amazonaws.com/human-pangenomics/T2T/CHM13/assemblies/chain/v1_nflo/hg19-chm13v2.chain")

In [4]:
gm12878_ctcf_chip_peak_bed = utils.lift_over(
    utils.save_file_from_url("https://www.encodeproject.org/files/ENCFF797SDL/@@download/ENCFF797SDL.bed.gz"),
    "bed",
    grch38_chm13v2_chain_file,
    chm13v2_file_id
)

Reading liftover chains
Mapping coordinates


lifted file: data/processed/ENCFF797SDL.t2t_chm13v2.bed
original: 41952; lifted: 41863; percent: 0.9978785278413425
total unmapped: 89
#Partially deleted in new
 69
#Deleted in new
 20


In [5]:
ctcf_motif_txt = utils.save_file_from_url("https://compbio.mit.edu/encode-motifs/matches.txt.gz")
ctcf_motif_txt = ctcf_motif_txt.rename(ctcf_motif_txt.with_suffix(".ctcf_motifs.txt"))

ctcf_motif_table = pd.read_csv(ctcf_motif_txt, sep=" ", header=None, names=["name", "chrom", "chromStart", "chromEnd", "strand"])
ctcf_motif_table = ctcf_motif_table.drop_duplicates(subset=["chrom", "chromStart", "chromEnd"])
ctcf_motif_table["name"] = "."
ctcf_motif_table["strand"] = "."
ctcf_motif_table["score"] = "."

utils.write_bed_file(ctcf_motif_table[["chrom", "chromStart", "chromEnd", "name", "score", "strand"]], config.processed_data_dir / ctcf_motif_txt.with_suffix(".bed").name)

ctcf_motif_bed = utils.lift_over(
    config.processed_data_dir / "matches.ctcf_motifs.bed",
    "bed",
    hg19_chm13v2_chain_file,
    chm13v2_file_id
)

Reading liftover chains
Mapping coordinates


lifted file: data/processed/matches.ctcf_motifs.t2t_chm13v2.bed
original: 97667053; lifted: 96978496; percent: 0.9929499562150196
total unmapped: 688557
#Deleted in new
 498403
#Partially deleted in new
 190104
#Split in new
 50


In [ ]:
atac_peak_file = utils.save_file_from_url("https://www.encodeproject.org/files/ENCFF748UZH/@@download/ENCFF748UZH.bed.gz")

In [ ]:
"gm12878_ctcf_chip_intersect_motif.top3k.bed"

# OPTIONAL: Redefine paths to avoid redownloading when rerunning later

In [4]:
gm12878_ctcf_chip_peak_bed = config.processed_data_dir / "ENCFF797SDL.t2t_chm13v2.bed"
ctcf_motif_bed = config.processed_data_dir / "matches.ctcf_motifs.t2t_chm13v2.bed"
t2t_reference = config.raw_data_dir / "chm13v2.0.fa"
# atac_peak_file = 

# Postprocessing
Perform any bespoke processing to produce final files for analysis and plotting

## CTCF

In [ ]:
ctcf_n_top_peaks = 3000
# NOTE: Matches window size set in CTCF_analysis.ipynb
ctcf_window_size = 1000

Select strong CTCF ChIP-seq peaks which overlap with a known CTCF motif, then find their center points.

In [ ]:
gm12878_CTCF_known_locations_file = config.processed_data_dir / "gm12878_ctcf_chip_intersect_motif.top3k.bed"
with gm12878_CTCF_known_locations_file.open("w") as fp:
    intersect_cmd = subprocess.Popen(
        [config.user_config["executables"]["bedtools_exe"], "intersect", "-u", "-a", gm12878_ctcf_chip_peak_bed, "-b", ctcf_motif_bed],
        stdout=subprocess.PIPE
    )
    sort_command = subprocess.Popen(
        ["sort", "-k", "7,7nr"],
        stdin=intersect_cmd.stdout, stdout=subprocess.PIPE
    )
    head_command = subprocess.Popen(
        ["head", "-n", "3000"],
        stdin=sort_command.stdout, stdout=fp, encoding="utf-8"
    )
    head_command.communicate()

known_location_table = utils.load_encode_narrowpeak_bed(gm12878_CTCF_known_locations_file)
known_location_table["chromStart"] = known_location_table["chromStart"] + ((known_location_table["chromEnd"] - known_location_table["chromStart"]) // 2)
known_location_table["chromEnd"] = known_location_table["chromStart"] + 1
utils.write_bed_file(known_location_table, gm12878_CTCF_known_locations_file)

print(f"output file: {gm12878_CTCF_known_locations_file.relative_to(config.working_dir)}")

output file: data/processed/gm12878_ctcf_chip_intersect_motif.top3k.bed


Define on-target CTCF sites as a window around each known location center point.

In [ ]:
gm12878_CTCF_on_target_file = config.processed_data_dir / "gm12878_ctcf.on_target.bed"
with gm12878_CTCF_on_target_file.open("w") as fp:
    subprocess.run(
        [config.user_config["executables"]["bedtools_exe"], "slop", "-i", gm12878_CTCF_known_locations_file, "-g", t2t_reference.with_suffix(".fa.fai"), "-b", str(ctcf_window_size)],
        stdout=fp, encoding="utf-8"
    )

Define off-target CTCF sites as flanking regions around these strong known CTCF locations

In [ ]:
gm12878_CTCF_off_target_file = config.processed_data_dir / "gm12878_ctcf.off_target.bed"
with gm12878_CTCF_off_target_file.open("w") as fp:
    subprocess.run(
        [config.user_config["executables"]["bedtools_exe"], "flank", "-i", gm12878_CTCF_on_target_file, "-g", t2t_reference.with_suffix(".fa.fai"), "-b", str(ctcf_window_size)],
        stdout=fp, encoding="utf-8"
    )